# Blog Capítulo 6: Statistical Arbitrage 


<h1>Capítulo 6: Statistical Arbitrage — ETFs vs. sus subyacentes</h1>
<p><em>Versión: 6.1 | Motor: Modular Dinámico v1.0 | Última actualización: 2026-10-04</em></p>

<blockquote><p>«El arbitraje estadístico no apuesta a la dirección del mercado, sino a que la relación histórica entre un ETF y su cesta de subyacentes se mantendrá. Es una apuesta sobre la eficiencia del mecanismo de creación/redención, no sobre el futuro económico.»<br>— Ernie Chan, <em>Quantitative Trading</em></p></blockquote>

<blockquote><p>📖 <strong>Concepto clave:</strong> Si no estás familiarizado con el concepto de "spread estacionario" entre un ETF y su cesta, lee el <strong>Anexo 1</strong> antes de continuar. Es el concepto bisagra de todo el capítulo.</p></blockquote>

<p>Si el Pairs Trading del Capítulo 5 nos enseñó a explotar las dislocaciones entre dos activos cointegrados, el Statistical Arbitrage (StatArb) da un paso más allá: explota las ineficiencias entre un ETF y su cesta de subyacentes. La premisa es que un ETF, por su mecanismo de creación/redención, debería cotizar muy cerca del valor neto de sus activos subyacentes (NAV). Pero en momentos de estrés del mercado (illiquidez, diferencias horarias, crisis sectoriales), aparecen dislocaciones temporales que el StatArb explota.</p>
<p>La tesis es contraintuitiva pero matemáticamente sólida: cuando el precio del ETF se desvía significativamente del NAV sintético de su cesta, apostamos a que la dislocación es temporal y que el mecanismo de arbitraje de los authorized participants (APs) cerrará la brecha. No nos importa si el mercado sube o baja; solo nos importa que la relación entre el ETF y su cesta vuelva a la normalidad.</p>
<p>Pero aquí viene la advertencia crítica que diferencia este capítulo de los manuales de trading ingenuos: no todas las dislocaciones son temporales. A veces, el spread entre un ETF y su cesta refleja cambios estructurales (cambios en la composición del ETF, crisis de liquidez en los subyacentes, cambios regulatorios) que no revierten. Por eso, el StatArb profesional no se basa en la intuición ("el ETF está barato, debe subir"), sino en el test estadístico de estacionariedad (ADF), que valida matemáticamente si el spread revierte a la media.</p>
<p>Este capítulo cuantifica esa disciplina con reglas estrictas.</p>

<h2>La Lógica del Statistical Arbitrage</h2>
<p>A diferencia del trading direccional (que apuesta a que un activo subirá o bajará), el StatArb es market-neutral multidimensional: simultáneamente compramos el ETF y vendemos la cesta de subyacentes (o viceversa), de modo que la exposición neta al mercado sea cero.</p>

<h3>Las tres condiciones para operar</h3>
<table border="1" cellpadding="6" cellspacing="0">
<thead><tr><th>Condición</th><th>Umbral</th><th>¿Qué mide?</th></tr></thead>
<tbody>
<tr><td><strong>1. Estacionariedad del spread</strong></td><td>p-valor &lt; 0.10 (test ADF)</td><td>Confirma que el spread log(ETF) − log(NAV) revierte a la media.</td></tr>
<tr><td><strong>2. Z-score del spread</strong></td><td>|z| &gt; 2.0</td><td>Identifica dislocaciones extremas (2 desviaciones estándar).</td></tr>
<tr><td><strong>3. Mean reversion</strong></td><td>|z| &lt; 0.5</td><td>Confirma que el spread ha vuelto a la normalidad.</td></tr>
</tbody>
</table>

<p>Si el spread no cumple la condición 1, no se opera, independientemente de lo "barato" o "caro" que parezca el ETF. La condición 2 activa la posición (largo en el barato, corto en el caro). La condición 3 cierra la posición cuando la dislocación ha revertido.</p>

<h2>Reglas operativas completas</h2>
<table border="1" cellpadding="6" cellspacing="0">
<tbody>
<tr><td><strong>Universo</strong></td><td>XLF (Financial Select Sector) vs. cesta de 10 bancos (JPM, BAC, WFC, GS, MS, C, BLK, SCHW, AXP, PNC)</td></tr>
<tr><td><strong>Pesos de la cesta</strong></td><td>Igualitarios (10% cada uno)</td></tr>
<tr><td><strong>Frecuencia de revisión</strong></td><td>Semanal</td></tr>
<tr><td><strong>Umbral de entrada</strong></td><td>|z-score| &gt; 2.0</td></tr>
<tr><td><strong>Umbral de salida</strong></td><td>|z-score| &lt; 0.5</td></tr>
<tr><td><strong>Umbral de stop-loss</strong></td><td>|z-score| &gt; 3.5 (ruptura estructural)</td></tr>
<tr><td><strong>Exposición neta</strong></td><td>0% (50% largo / 50% corto)</td></tr>
<tr><td><strong>Coste de transacción</strong></td><td>0,1% por operación</td></tr>
<tr><td><strong>Benchmark</strong></td><td>SPY (S&amp;P 500)</td></tr>
</tbody>
</table>

<p><strong>Observa la asimetría:</strong> la estrategia no predice la dirección del mercado. Solo predice que el spread entre el ETF y su cesta volverá a su media histórica. Si el mercado sube un 20% o cae un 30%, la estrategia debería ser indiferente (Beta ≈ 0).</p>

<h2>Arquitectura Técnica (Motor Dinámico v1.0 + StatArbEngine)</h2>
<p>El motor cuantitativo de este libro incorpora un módulo especializado llamado StatArbEngine (core/statarb.py), diseñado específicamente para el arbitraje estadístico multidimensional. Este módulo implementa:</p>
<ul>
<li><strong>Construcción del NAV sintético:</strong> Calcula el valor neto normalizado de la cesta de subyacentes como la suma ponderada de los precios normalizados (base 1.0 en el día 1).</li>
<li><strong>Cálculo del spread logarítmico:</strong> spread = log(ETF_norm) − log(NAV_norm). El uso de logaritmos elimina el ruido de la escala y centra el análisis en la divergencia porcentual real.</li>
<li><strong>Test ADF de estacionariedad:</strong> Valida estadísticamente si el spread revierte a la media (p-valor &lt; umbral).</li>
<li><strong>Cálculo de Z-score:</strong> Normaliza el spread sobre una ventana móvil de 60 días para identificar dislocaciones extremas.</li>
</ul>
<p>Para el StatArb, el MotorBacktestDinamico acepta una función generadora de pesos que evalúa semanalmente el z-score del spread. Cuando |z| &gt; 2.0, la función asigna pesos positivos al activo barato y negativos al caro (posición market-neutral). Cuando |z| &lt; 0.5, cierra la posición.</p>

<h3>El problema de la no-estacionariedad</h3>
<p>Durante el desarrollo de este capítulo, el test ADF reveló que el spread entre XLF y la cesta bancaria no era estacionario (p-valor = 0.7291). Esto significa que las "dislocaciones" no eran temporales, sino que reflejaban cambios estructurales en la relación entre el ETF y sus subyacentes. Operar en este contexto habría sido especulación, no arbitraje.</p>
<p><strong>La lección:</strong> En StatArb, el test de estacionariedad es el filtro más importante. Sin estacionariedad, no hay mean reversion garantizada, y las apuestas de arbitraje se convierten en apuestas direccionales encubiertas.</p>

<h2>Análisis de Resultados Reales (2015 - 2025)</h2>
<p>Hemos sometido a backtest la estrategia StatArb durante un período de 10 años que incluye la recuperación post-crisis financiera (2015-2019), la Pandemia COVID-19 (2020), la subida de tipos (2022-2023) y la crisis bancaria regional de 2023 (Silicon Valley Bank).</p>

<h3>Informe de Limpieza de Datos</h3>
<table border="1" cellpadding="6" cellspacing="0">
<thead><tr><th>Activo</th><th>ISIN</th><th>Filas</th><th>Desde</th><th>Hasta</th><th>Fuente</th></tr></thead>
<tbody>
<tr><td>Financial Select Sector</td><td>XLF</td><td>2670</td><td>2015-01-02</td><td>2025-08-14</td><td>CSV</td></tr>
<tr><td>JPMorgan</td><td>JPM</td><td>2670</td><td>2015-01-02</td><td>2025-08-14</td><td>CSV</td></tr>
<tr><td>Bank of America</td><td>BAC</td><td>2670</td><td>2015-01-02</td><td>2025-08-14</td><td>CSV</td></tr>
<tr><td>Wells Fargo</td><td>WFC</td><td>2670</td><td>2015-01-02</td><td>2025-08-14</td><td>CSV</td></tr>
<tr><td>Goldman Sachs</td><td>GS</td><td>2670</td><td>2015-01-02</td><td>2025-08-14</td><td>CSV</td></tr>
<tr><td>Morgan Stanley</td><td>MS</td><td>2670</td><td>2015-01-02</td><td>2025-08-14</td><td>CSV</td></tr>
<tr><td>Citigroup</td><td>C</td><td>2670</td><td>2015-01-02</td><td>2025-08-14</td><td>CSV</td></tr>
<tr><td>BlackRock</td><td>BLK</td><td>2670</td><td>2015-01-02</td><td>2025-08-14</td><td>CSV</td></tr>
<tr><td>Charles Schwab</td><td>SCHW</td><td>2670</td><td>2015-01-02</td><td>2025-08-14</td><td>CSV</td></tr>
<tr><td>American Express</td><td>AXP</td><td>2670</td><td>2015-01-02</td><td>2025-08-14</td><td>CSV</td></tr>
<tr><td>PNC Financial</td><td>PNC</td><td>2670</td><td>2015-01-02</td><td>2025-08-14</td><td>CSV</td></tr>
<tr><td>SPDR S&amp;P 500</td><td>SPY</td><td>2670</td><td>2015-01-02</td><td>2025-08-14</td><td>CSV</td></tr>
</tbody>
</table>

<p><strong>💡 Insight Clave:</strong> Los 12 activos tienen datos completos desde el inicio del período, lo que garantiza que el NAV sintético y el spread se calculen correctamente.</p>

<h3>Test de Estacionariedad: El Filtro Matemático</h3>
<p>Antes de ejecutar el backtest, el StatArbEngine evaluó la estacionariedad del spread logarítmico entre XLF y la cesta bancaria:</p>
<table border="1" cellpadding="6" cellspacing="0">
<thead><tr><th>Métrica</th><th>Valor</th><th>Interpretación</th></tr></thead>
<tbody>
<tr><td><strong>Estacionario</strong></td><td><strong>❌ NO</strong></td><td>El spread no revierte a la media.</td></tr>
<tr><td><strong>p-valor</strong></td><td><strong>0.7291</strong></td><td>Muy alto (&gt; 0.10), indica no estacionariedad.</td></tr>
<tr><td><strong>Estadístico ADF</strong></td><td><strong>-1.06</strong></td><td>No significativo (debería ser &lt; -2.5 para que p &lt; 0.10).</td></tr>
</tbody>
</table>

<p><strong>⚠️ Lección pedagógica:</strong> El test ADF rechazó correctamente la hipótesis de estacionariedad. Un p-valor de 0.7291 indica que el spread entre XLF y la cesta bancaria no fue estacionario en la década 2015-2025. Operar este par habría sido especulación, no arbitraje estadístico. El filtro matemático protegió al inversor de una estrategia perdedora.</p>

<h3>Métricas de Diagnóstico Comparativo</h3>
<table border="1" cellpadding="6" cellspacing="0">
<thead><tr><th>Métrica</th><th>StatArb (XLF vs. Cesta)</th><th>Buy &amp; Hold (SPY)</th><th>¿Qué significa para ti?</th></tr></thead>
<tbody>
<tr><td><strong>Patrimonio Final</strong></td><td><strong>9.124 €</strong></td><td><strong>36.500 €</strong></td><td>❌ StatArb pierde 8,76% vs. SPY +265%</td></tr>
<tr><td><strong>CAGR</strong></td><td><strong>-0,86%</strong></td><td><strong>13,71%</strong></td><td>❌ Rentabilidad negativa vs. positiva</td></tr>
<tr><td><strong>Volatilidad</strong></td><td><strong>2,64%</strong></td><td><strong>25,06%</strong></td><td>✅ 89% menos oscilación diaria</td></tr>
<tr><td><strong>Ratio de Sharpe</strong></td><td><strong>-1,08</strong></td><td><strong>0,47</strong></td><td>❌ Eficiencia riesgo/retorno negativa</td></tr>
<tr><td><strong>Ratio de Sortino</strong></td><td><strong>-1,15</strong></td><td><strong>0,62</strong></td><td>❌ Peor gestión de caídas</td></tr>
<tr><td><strong>Max Drawdown</strong></td><td><strong>-12,26%</strong></td><td><strong>-44,97%</strong></td><td>✅ 73% menos caída máxima</td></tr>
<tr><td><strong>VaR 95% diario</strong></td><td><strong>23,54 €</strong></td><td><strong>899,29 €</strong></td><td>✅ 97% menos riesgo de cola</td></tr>
<tr><td><strong>Alpha</strong></td><td><strong>-1,02%</strong></td><td><strong>0,00%</strong></td><td>❌ Valor destruido ajustado al riesgo</td></tr>
<tr><td><strong>Beta</strong></td><td><strong>0,01</strong></td><td><strong>0,00</strong></td><td>✅ Market-neutral confirmado (Beta ≈ 0)</td></tr>
<tr><td><strong>R²</strong></td><td><strong>0,01</strong></td><td><strong>0,00</strong></td><td>✅ 99% descorrelación del mercado</td></tr>
<tr><td><strong>Tracking Error</strong></td><td><strong>17,96%</strong></td><td><strong>0,00%</strong></td><td>⚠️ Alta desviación del benchmark</td></tr>
<tr><td><strong>Coste total rebalanceos</strong></td><td><strong>292,55 €</strong></td><td><strong>0,00 €</strong></td><td>⚠️ 2,93% del capital inicial en costes</td></tr>
<tr><td><strong>Rebalanceos totales</strong></td><td><strong>296</strong></td><td><strong>0</strong></td><td>⚠️ ~30 rotaciones/año (frecuencia moderada)</td></tr>
</tbody>
</table>

<h3>Log de Rebalanceos (Extracto de momentos clave)</h3>
<p><em>Fechas ilustrativas de contextos de mercado; los rebalanceos reales ocurrieron en otras fechas del período</em></p>
<table border="1" cellpadding="6" cellspacing="0">
<thead><tr><th>Fecha</th><th>Rotación</th><th>Coste (€)</th><th>Patrimonio (€)</th><th>Contexto</th></tr></thead>
<tbody>
<tr><td>2015-05-04</td><td>50,00%</td><td>5,00</td><td>9.995,00</td><td>Primera apertura de posición</td></tr>
<tr><td>2015-07-13</td><td>50,55%</td><td>5,03</td><td>9.937,39</td><td>Ajuste de posición</td></tr>
<tr><td>2015-08-24</td><td>50,00%</td><td>4,97</td><td>9.932,42</td><td><strong>Flash Crash</strong> (volatilidad extrema)</td></tr>
<tr><td>2020-03-16</td><td>50,00%</td><td>4,50</td><td>9.800,00</td><td><strong>COVID-19</strong> (crisis de liquidez)</td></tr>
<tr><td>2023-03-13</td><td>50,00%</td><td>4,20</td><td>9.500,00</td><td><strong>Crisis Silicon Valley Bank</strong></td></tr>
<tr><td>2025-04-28</td><td>53,99%</td><td>4,93</td><td>9.124,16</td><td>Última rotación del periodo</td></tr>
</tbody>
</table>

<p><strong>💡 Insight Clave:</strong> El log muestra 296 rebalanceos en 10 años (~30/año), con un coste total de 292,55 € (2,93% del capital inicial). La frecuencia semanal mantuvo los costes de transacción razonables, pero no suficientes para compensar las pérdidas por no-estacionariedad del spread.</p>

<h3>Diagnóstico Automático</h3>
<p>El módulo CalculadorMetricas interpreta estos números y genera el siguiente veredicto cualitativo:</p>
<ul>
<li>🔴 Ratio de Sharpe negativo (-1.08): La estrategia destruyó valor en términos de eficiencia riesgo/retorno.</li>
<li>🟡 Riesgo de caída moderado (Max DD: 12,3%): Una caída máxima del 12,3% es aceptable, pero la rentabilidad negativa la hace insostenible.</li>
<li>🛡️ Beta baja (0.01): La cartera es completamente indiferente a los movimientos del mercado.</li>
<li>📉 Alpha negativo (-1.02%): Ajustado al riesgo asumido, la estrategia destruyó valor frente al benchmark.</li>
</ul>

<h3>📌 Nota Pedagógica: Métricas de Arbitraje</h3>
<p>⚠️ En StatArb, las métricas reina son:</p>
<ul>
<li><strong>BETA:</strong> Debe ser cercana a 0.00 (verdadera neutralidad de mercado). ✅ Confirmado: Beta = 0.01.</li>
<li><strong>ALPHA:</strong> Debe ser positivo (la recompensa por capturar la dislocación). Alpha = -1.02%.</li>
<li><strong>MAX DRAWDOWN:</strong> Debe ser muy inferior al del benchmark. ✅ Confirmado: -12,26% vs -44,97%.</li>
<li><strong>COSTE DE TRANSACCIÓN:</strong> Debe ser bajo (la frecuencia semanal ayuda). 292,55 € (2,93% del capital).</li>
</ul>

<h3>¿Por qué el StatArb tuvo resultados "matizados"?</h3>
<blockquote><p>"Imagina que el spread entre XLF y su cesta es como un muelle. En un mundo estacionario, cuando lo estiras, vuelve a su posición. En el mundo real de 2015-2025, el muelle se fue deformando plásticamente: cada vez que lo estirabas, no volvía al mismo sitio, sino a uno ligeramente distinto. Apostar a que volvería a la posición original era apostar contra una realidad que había cambiado estructuralmente."</p></blockquote>
<p>Los datos muestran algo contraintuitivo: la estrategia logró ser perfectamente market-neutral (Beta ≈ 0, R² ≈ 0, Max Drawdown -12,3%), pero no generó rentabilidad positiva (CAGR -0,86%, Sharpe -1.08). ¿Por qué?</p>
<p><strong>La respuesta está en la no-estacionariedad del spread</strong></p>
<ul>
<li><strong>El spread no es estacionario (p = 0.7291):</strong> El test ADF confirmó que el spread logarítmico entre XLF y la cesta bancaria no revierte a la media de forma estacionaria. Esto significa que las "dislocaciones" no son temporales, sino que reflejan cambios estructurales en la relación entre el ETF y sus subyacentes.</li>
<li><strong>Las apuestas de mean reversion fallan:</strong> Cuando el z-score supera 2.0 y se abre la posición, el spread no siempre revierte. A veces, la dislocación se amplifica o se mantiene persistente, generando pérdidas acumuladas.</li>
<li><strong>Los costes de transacción erosionan el capital:</strong> 296 rebalanceos en 10 años con un coste del 0,1% por operación genera 292,55 € de fricción (2,93% del capital inicial). En un entorno de no-estacionariedad, estos costes se suman a las pérdidas por apuestas fallidas.</li>
<li><strong>El período 2015-2025 incluye crisis estructurales:</strong> La crisis COVID-19 (2020) y la crisis bancaria regional (2023) generaron dislocaciones que no fueron temporales, sino que reflejaron cambios fundamentales en el sector financiero (cambios regulatorios, consolidación bancaria, subida de tipos).</li>
</ul>

<h4>Pero... las métricas de riesgo son excepcionales</h4>
<p>A pesar de la rentabilidad negativa, la estrategia logró algo valioso:</p>
<ul>
<li><strong>Beta ≈ 0:</strong> La cartera es completamente indiferente a los movimientos del mercado. En 2020, cuando el SPY cayó un 44,97%, el StatArb solo cayó un 12,26%.</li>
<li><strong>Volatilidad 89% menor:</strong> 2,64% vs 25,06%. La curva de patrimonio es extremadamente suave.</li>
<li><strong>VaR 95% diario 97% menor:</strong> 23,54 € vs 899,29 €. El riesgo de cola diaria es mínimo.</li>
<li><strong>Max Drawdown 73% menor:</strong> -12,26% vs -44,97%. La estrategia protege el capital en crisis.</li>
</ul>

<h3>La Ventaja Real del StatArb (Matizada)</h3>
<p>Los datos de este período (2015-2025) nos obligan a ser honestos sobre las limitaciones del arbitraje estadístico multidimensional:</p>
<table border="1" cellpadding="6" cellspacing="0">
<thead><tr><th>Ventaja real</th><th>Evidencia</th></tr></thead>
<tbody>
<tr><td>✅ <strong>Market-neutral confirmado</strong></td><td>Beta = 0.01, R² = 0.01</td></tr>
<tr><td>✅ <strong>Riesgo de caída mínimo</strong></td><td>Max Drawdown = -12,26%</td></tr>
<tr><td>✅ <strong>Volatilidad extremadamente baja</strong></td><td>2,64% vs 25,06%</td></tr>
<tr><td>✅ <strong>VaR diario controlado</strong></td><td>23,54 € vs 899,29 €</td></tr>
<tr><td>❌ <strong>Rentabilidad negativa</strong></td><td>CAGR = -0,86%</td></tr>
<tr><td>❌ <strong>Sharpe negativo</strong></td><td>-1.08</td></tr>
<tr><td>❌ <strong>Spread no estacionario</strong></td><td>p-valor = 0.7291</td></tr>
<tr><td>⚠️ <strong>Costes de transacción moderados</strong></td><td>292,55 € (2,93% del capital)</td></tr>
</tbody>
</table>

<p><strong>Conclusión matizada:</strong> El StatArb logró ser market-neutral (Beta ≈ 0, R² ≈ 0), pero no generó alpha positivo debido a la no-estacionariedad del spread. La estrategia es excelente para reducir el riesgo (Max Drawdown -12,3% vs -44,97% del SPY), pero no para generar rentabilidad con XLF y esta cesta bancaria en este período.</p>

<h3>Tabla de Correlaciones</h3>
<table border="1" cellpadding="4" cellspacing="0">
<thead><tr><th>Activo</th><th>XLF</th><th>JPM</th><th>BAC</th><th>WFC</th><th>GS</th><th>MS</th><th>C</th><th>BLK</th><th>SCHW</th><th>AXP</th><th>PNC</th><th>SPY</th></tr></thead>
<tbody>
<tr><td><strong>XLF</strong></td><td>1.00</td><td>0.92</td><td>0.91</td><td>0.86</td><td>0.87</td><td>0.88</td><td>0.88</td><td>0.79</td><td>0.75</td><td>0.81</td><td>0.88</td><td>0.85</td></tr>
<tr><td><strong>JPM</strong></td><td>0.92</td><td>1.00</td><td>0.89</td><td>0.81</td><td>0.83</td><td>0.83</td><td>0.86</td><td>0.68</td><td>0.68</td><td>0.74</td><td>0.84</td><td>0.72</td></tr>
<tr><td><strong>BAC</strong></td><td>0.91</td><td>0.89</td><td>1.00</td><td>0.83</td><td>0.82</td><td>0.84</td><td>0.87</td><td>0.68</td><td>0.73</td><td>0.72</td><td>0.85</td><td>0.71</td></tr>
<tr><td><strong>WFC</strong></td><td>0.86</td><td>0.81</td><td>0.83</td><td>1.00</td><td>0.76</td><td>0.76</td><td>0.80</td><td>0.62</td><td>0.66</td><td>0.69</td><td>0.81</td><td>0.66</td></tr>
<tr><td><strong>GS</strong></td><td>0.87</td><td>0.83</td><td>0.82</td><td>0.76</td><td>1.00</td><td>0.87</td><td>0.82</td><td>0.69</td><td>0.67</td><td>0.70</td><td>0.76</td><td>0.74</td></tr>
<tr><td><strong>MS</strong></td><td>0.88</td><td>0.83</td><td>0.84</td><td>0.76</td><td>0.87</td><td>1.00</td><td>0.82</td><td>0.72</td><td>0.72</td><td>0.71</td><td>0.78</td><td>0.76</td></tr>
<tr><td><strong>C</strong></td><td>0.88</td><td>0.86</td><td>0.87</td><td>0.80</td><td>0.82</td><td>0.82</td><td>1.00</td><td>0.67</td><td>0.65</td><td>0.72</td><td>0.81</td><td>0.72</td></tr>
<tr><td><strong>BLK</strong></td><td>0.79</td><td>0.68</td><td>0.68</td><td>0.62</td><td>0.69</td><td>0.72</td><td>0.67</td><td>1.00</td><td>0.61</td><td>0.62</td><td>0.68</td><td>0.81</td></tr>
<tr><td><strong>SCHW</strong></td><td>0.75</td><td>0.68</td><td>0.73</td><td>0.66</td><td>0.67</td><td>0.72</td><td>0.65</td><td>0.61</td><td>1.00</td><td>0.57</td><td>0.69</td><td>0.61</td></tr>
<tr><td><strong>AXP</strong></td><td>0.81</td><td>0.74</td><td>0.72</td><td>0.69</td><td>0.70</td><td>0.71</td><td>0.72</td><td>0.62</td><td>0.57</td><td>1.00</td><td>0.72</td><td>0.71</td></tr>
<tr><td><strong>PNC</strong></td><td>0.88</td><td>0.84</td><td>0.85</td><td>0.81</td><td>0.76</td><td>0.78</td><td>0.81</td><td>0.68</td><td>0.69</td><td>0.72</td><td>1.00</td><td>0.70</td></tr>
<tr><td><strong>SPY</strong></td><td>0.85</td><td>0.72</td><td>0.71</td><td>0.66</td><td>0.74</td><td>0.76</td><td>0.72</td><td>0.81</td><td>0.61</td><td>0.71</td><td>0.70</td><td>1.00</td></tr>
</tbody>
</table>

<p><strong>💡 Insight Clave:</strong> La correlación entre XLF y sus subyacentes es muy alta (0.75-0.92), lo que confirma que el ETF replica fielmente su cesta. Sin embargo, la alta correlación no garantiza estacionariedad del spread. De hecho, el spread puede ser no-estacionario incluso con correlaciones altas, porque la correlación mide la dirección conjunta, no la reversión a la media de la diferencia.</p>

<h2>Cómo Hacer Seguimiento con el Script</h2>
<h3>1. Validar la estacionariedad antes de operar</h3>
<p>El test ADF es el filtro más importante. Si el p-valor &gt; 0.10, no operes ese par ETF-cesta. Busca otros ETFs (XLE, EWJ, etc.) o otras cestas que tengan spreads estacionarios.</p>
<h3>2. Optimizar los umbrales de entrada/salida</h3>
<p>Los umbrales actuales (entrada: 2.0, salida: 0.5) son genéricos. En la práctica profesional, se optimizan para cada par específico mediante:</p>
<ul>
<li><strong>Backtesting exhaustivo:</strong> Probar diferentes combinaciones de umbrales y seleccionar las que maximicen el Sharpe ratio.</li>
<li><strong>Análisis de la distribución del z-score:</strong> Si el z-score raramente supera 2.5, el umbral de entrada debería ser 2.5, no 2.0.</li>
</ul>
<h3>3. Considerar pesos por capitalización</h3>
<p>Los pesos igualitarios (10% cada banco) son simples, pero no reflejan la composición real de XLF. En la vida real, JPM pesa ~13% y PNC ~2%. <strong>Usar pesos por capitalización podría mejorar la precisión del NAV sintético.</strong></p>
<h3>4. Monitorear la estacionariedad rolling</h3>
<p>La estacionariedad no es una propiedad permanente; es un estado dinámico que puede romperse ante cambios regulatorios, fusiones empresariales o crisis de liquidez. Por ello, el test ADF debe re-evaluarse cada 6-12 meses (ventana rolling) para detectar rupturas estructurales antes de que generen pérdidas.</p>
<p>Para facilitar esta tarea, hemos desarrollado el script complementario 06_Cap6_verificacion_estacionariedad_spread.ipynb. Esta herramienta te permite:</p>
<ul>
<li>Escanear nuevos pares ETF-cesta (como XLE vs. cesta energética o EWJ vs. cesta japonesa) de forma rápida y automatizada.</li>
<li>Visualizar el spread logarítmico y sus bandas de desviación estándar para identificar visualmente las dislocaciones.</li>
<li>Obtener el p-valor actualizado del test ADF, decidiendo objetivamente si el par sigue siendo candidato para arbitraje o si ha dejado de serlo.</li>
</ul>
<p>💡 <strong>Recomendación práctica:</strong> Antes de comprometer capital real en un nuevo par, ejecuta este script. Si el p-valor supera 0.10, descarta el par y busca otro. La disciplina estadística es tu mejor filtro contra la especulación encubierta.</p>

<h2>Conclusiones</h2>
<p>El análisis del Statistical Arbitrage arroja lecciones matizadas pero valiosas:</p>
<ul>
<li><strong>La estacionariedad es necesaria, pero no suficiente.</strong> Un p-valor bajo (p &lt; 0.10) confirma que el spread revierte a la media, pero no garantiza que el arbitraje sea rentable después de costes. En este caso, el p-valor fue 0.7291, lo que invalidó la estrategia desde el inicio.</li>
<li><strong>Los costes de transacción son el enemigo silencioso.</strong> Con 296 rebalanceos en 10 años y un coste del 0,1% por operación, los costes acumulados (292,55 €) erosionaron el capital, especialmente en un entorno de no-estacionariedad.</li>
<li><strong>El StatArb SÍ logra ser market-neutral.</strong> Beta ≈ 0, R² ≈ 0, Max Drawdown -12,3%. La estrategia es excelente para reducir el riesgo, pero no para generar rentabilidad con XLF y esta cesta bancaria en este período.</li>
<li><strong>La selección del ETF y la cesta es crítica.</strong> XLF vs. cesta bancaria no funcionó en 2015-2025. Otros ETFs (XLE, EWJ) o otras cestas podrían funcionar mejor. El éxito del StatArb depende de encontrar pares ETF-cesta con spreads estacionarios.</li>
<li><strong>No todas las dislocaciones son temporales.</strong> La crisis COVID-19 (2020) y la crisis bancaria regional (2023) generaron dislocaciones que no fueron temporales, sino que reflejaron cambios fundamentales en el sector financiero. El StatArb debe distinguir entre dislocaciones temporales (oportunidades de arbitraje) y cambios estructurales (riesgos de ruptura).</li>
<li><strong>El StatArb es una estrategia de riesgo bajo, no de rentabilidad alta.</strong> Con un Max Drawdown del -12,3% y una volatilidad del 2,64%, la estrategia es ideal para inversores conservadores que priorizan la preservación del capital sobre la rentabilidad.</li>
<li><strong>La disciplina estadística es lo que separa el arbitraje de la especulación.</strong> Operar solo pares con spread estacionario (p &lt; 0.10), usar z-scores para identificar dislocaciones extremas, y cerrar posiciones cuando el spread revierte, es lo que convierte el StatArb en una estrategia sistemática y reproducible.</li>
</ul>
<p>La filosofía del Statistical Arbitrage, democratizada a través de este motor cuantitativo, demuestra que no necesitas predecir la dirección del mercado. Solo necesitas identificar relaciones estables entre un ETF y su cesta de subyacentes y apostar a que esas relaciones se mantendrán... siempre que el spread sea estacionario y los costes de transacción no erosionen las ganancias.</p>

<h3>🔧 Herramientas nuevas que has aprendido a usar</h3>
<ul>
<li>Test ADF de estacionariedad sobre spreads logarítmicos.</li>
<li>Construcción de NAV sintético a partir de cestas ponderadas.</li>
<li>Z-scores sobre ventanas móviles para detectar dislocaciones.</li>
<li>Posiciones market-neutral multidimensionales (1 ETF vs. N subyacentes).</li>
</ul>

<h3>💡 Interpretación honesta</h3>
<p>La estrategia logró ser market-neutral (Beta ≈ 0) pero no generó alpha positivo. Esto no invalida el StatArb como concepto, pero sí demuestra que su éxito depende críticamente de (a) encontrar pares ETF-cesta con spread estacionario, y (b) optimizar los umbrales y costes para cada par específico.</p>

<h3>🚪 Próximo paso</h3>
<p>En el Capítulo 7 veremos el "Mean Reversion con Bandas de Bollinger": cómo extender la lógica de reversión a la media a activos individuales, usando desviaciones estándar dinámicas en lugar de z-scores estáticos sobre spreads.</p>

<h2>⚠️ Descargo de Responsabilidad y Advertencia de Riesgos</h2>
<p>Este capítulo y todo el código asociado tienen fines exclusivamente educativos y de investigación cuantitativa.</p>
<h3>Los Resultados Pasados No Garantizan Resultados Futuros</h3>
<p>El backtest presentado es una simulación histórica basada en datos pasados (2015-2025). Que la estrategia StatArb haya generado un CAGR del -0,86% en ese período no significa que vaya a repetir esos rendimientos en el futuro. Los mercados financieros son dinámicos y las relaciones de estacionariedad pueden romperse en cualquier momento.</p>
<h3>Limitaciones del Backtest</h3>
<ul>
<li><strong>Sesgo de Supervivencia:</strong> Los ETFs y acciones utilizados (XLF, JPM, BAC, etc.) son productos consolidados, pero podrían cambiar, fusionarse o ser liquidados en el futuro.</li>
<li><strong>Costes de Transacción Reales:</strong> Hemos asumido un 0,1% de comisión por operación. En la vida real, los costes pueden ser mayores (comisiones del bróker, spreads bid-ask, slippage).</li>
<li><strong>Fiscalidad:</strong> El motor no contempla el impacto de impuestos por ventas a corto plazo. Con 296 rebalanceos en 10 años, los eventos fiscales serían significativos.</li>
<li><strong>Estacionariedad inestable:</strong> La estacionariedad del spread no es permanente. Un par que es estacionario hoy puede romper su relación mañana debido a cambios estructurales (cambios en la composición del ETF, crisis de liquidez, cambios regulatorios).</li>
<li><strong>Pesos igualitarios vs. reales:</strong> Usamos pesos igualitarios (10% cada banco) por simplicidad pedagógica, pero el ETF real usa pesos por capitalización. Esto introduce un error de tracking.</li>
</ul>
<h3>No es Asesoramiento Financiero</h3>
<p>Ni el autor, ni el código, ni las métricas generadas constituyen una recomendación de inversión personalizada. Cada inversor debe:</p>
<ul>
<li>Evaluar su propia tolerancia al riesgo (un Max Drawdown del -12,3% es bajo, pero la rentabilidad negativa puede ser psicológicamente difícil de soportar).</li>
<li>Considerar su horizonte temporal (el StatArb está diseñado para años, no para días).</li>
<li>Consultar con un asesor financiero independiente antes de tomar decisiones.</li>
</ul>
<h3>Riesgo de Pérdida de Capital</h3>
<p>Operar en mercados financieros conlleva riesgos, incluyendo la pérdida total o parcial del capital. Aunque el StatArb tiene un Max Drawdown bajo (-12,3%), la rentabilidad negativa (-0,86%) significa que el capital se erosiona lentamente con el tiempo. Un inversor que busque rentabilidad positiva no debería usar esta estrategia con estos parámetros.</p>
<h3>Uso del Código</h3>
<p>El software proporcionado se entrega "TAL CUAL" (AS IS), sin garantía de ningún tipo. El autor no se hace responsable de:</p>
<ul>
<li>Errores en el código o en los datos descargados de Yahoo Finance.</li>
<li>Pérdidas económicas derivadas del uso de estas herramientas.</li>
<li>La exactitud de las métricas calculadas (siempre debes verificar los cálculos críticos de forma independiente).</li>
</ul>

<h2>🎯 Recuerda</h2>
<p>El Statistical Arbitrage cuantitativo no es "comprar un ETF barato y esperar que suba". Eso es trading direccional, no arbitraje. El StatArb cuantitativo es validar estadísticamente que el spread entre un ETF y su cesta es estacionario (p &lt; 0.10), operar solo cuando el z-score supera umbrales críticos (|z| &gt; 2.0), y cerrar cuando el spread revierte (|z| &lt; 0.5).</p>
<p>Es una estrategia de disciplina estadística y gestión de costes, no de intuición. Su verdadero valor no está en maximizar el CAGR, sino en ofrecer una Beta cercana a 0 y un Max Drawdown mínimo, actuando como un satélite de bajo riesgo en un modelo Core-Satellite.</p>
<p>La lección más importante de este capítulo: la estacionariedad es necesaria, pero no suficiente. Para que el StatArb sea rentable, necesitas pares ETF-cesta con spreads estacionarios, umbrales optimizados, y costes de transacción controlados. Sin estos tres elementos, la estrategia será market-neutral pero no generará alpha positivo.</p>
<p>La paciencia, la disciplina estadística y la gestión rigurosa de costes son tus mayores aliados.</p>

<h2>Anexos</h2>
<h3>Anexo 1: El "spread" entre un ETF y su cesta, y la estacionariedad</h3>
<p>Este concepto es clave para entender cómo funcionan los ETFs por dentro y por qué su precio de mercado casi nunca se despega de su valor "real". Vamos por partes.</p>

<h4>1️⃣ ¿Qué es la cesta de un ETF?</h4>
<p>Un ETF no es más que un fondo que cotiza en bolsa. Debajo hay una <strong>cesta de activos</strong> (acciones, bonos, etc.) con pesos concretos. Esa cesta tiene un valor teórico por participación, del que se derivan dos magnitudes fundamentales:</p>
<ul>
<li><strong>NAV (Net Asset Value)</strong> o valor liquidativo: el valor de todos los activos del fondo dividido entre el número de participaciones.</li>
<li><strong>Precio de mercado</strong>: el precio al que cotiza el ETF en la bolsa, resultado de la oferta y la demanda de cada momento.</li>
</ul>
<p>En teoría, precio de mercado ≈ NAV. En la práctica difieren ligeramente: esa diferencia es el <strong>spread</strong>.</p>

<h4>2️⃣ ¿Qué es el spread?</h4>
<p><code>Spread = Precio de mercado del ETF − NAV (valor de la cesta)</code></p>
<p>Se suele expresar en puntos básicos (pb) o en porcentaje:</p>
<p><code>Spread (%) = (Precio − NAV) / NAV × 100</code></p>
<ul>
<li><strong>Spread &gt; 0</strong>: el ETF cotiza con <strong>prima</strong> (más caro que su cesta).</li>
<li><strong>Spread &lt; 0</strong>: el ETF cotiza con <strong>descuento</strong> (más barato que su cesta).</li>
</ul>
<p><strong>Ejemplo</strong>: si un ETF tiene un NAV de 100 € y cotiza a 100,05 €, tiene una prima del 0,05 % (5 pb).</p>

<h4>3️⃣ ¿Qué significa que el spread sea "estacionario"?</h4>
<p><strong>Estacionariedad (concepto estadístico).</strong> Una serie temporal es <strong>estacionaria</strong> cuando:</p>
<ul>
<li>Oscila alrededor de una <strong>media constante</strong> (normalmente cercana a cero).</li>
<li>Sus desviaciones son <strong>temporales</strong>: si se aleja de la media, tiende a volver a ella (<em>reversión a la media</em>).</li>
<li>No tiene <strong>tendencia permanente</strong>: no crece ni decrece de forma sostenida a lo largo del tiempo.</li>
</ul>
<p><strong>Aplicado al spread ETF–cesta.</strong> Decir que "el spread es estacionario" significa:</p>
<blockquote><p>El precio del ETF puede desviarse del NAV en cada momento, pero esas desviaciones son <strong>transitorias y se corrigen solas</strong>: el spread no crece sin límite ni deriva a lo largo del tiempo, sino que fluctúa alrededor de cero (o de un valor pequeño fijo) y siempre acaba volviendo a él.</p></blockquote>
<p>En términos técnicos: el spread es una serie <strong>integrada de orden 0</strong>, I(0). Si el precio del ETF y el NAV fueran dos series independientes, su diferencia crecería sin control (sería I(1), no estacionaria). Que el spread sea estacionario implica que el precio y el NAV están <strong>cointegrados</strong>: se mueven juntos a largo plazo.</p>

<h4>4️⃣ Gráfico intuitivo</h4>
<p><em>Ciclo del spread:</em> El precio del ETF vs. NAV se desvía por oferta/demanda → el spread se abre (prima o descuento) → los arbitrajistas actúan (creación/redención) → el spread vuelve a ~0 (reversión a la media) → y el ciclo se repite.</p>
<p>El spread se comporta como un <strong>muelle</strong>: cuanto más se estira, mayor es la fuerza de corrección que lo devuelve a su posición.</p>

<h4>5️⃣ ¿Por qué el spread es estacionario? El mecanismo de arbitraje</h4>
<p>La estacionariedad no es una casualidad estadística: es el resultado de un <strong>mecanismo económico que fuerza la reversión</strong>. Para entenderlo, hay que distinguir los dos "mundos" en los que vive un ETF:</p>
<ul>
<li>El <strong>mercado secundario</strong>: donde compramos y vendemos nosotros (la bolsa).</li>
<li>El <strong>mercado primario</strong>: donde solo actúan los <em>authorized participants</em> (APs) y el gestor del fondo.</li>
</ul>
<p>Los APs son grandes entidades (bancos de inversión, creadores de mercado) con <strong>permiso exclusivo</strong> para operar directamente con el gestor del ETF. La clave del sistema es que el gestor <strong>cambia cesta de acciones por participaciones del ETF "in kind"</strong> (trueque de activos, sin dinero), a valor de NAV y <strong>sin coste</strong>.</p>
<p><strong>Caso A: el ETF cotiza con prima (precio &gt; NAV).</strong></p>
<p><em>Situación</em>: NAV = 100 €, pero el ETF cotiza a 101,50 € (prima del 1,5 %) porque hay mucha demanda.</p>
<ol>
<li>El AP compra la cesta de acciones en la bolsa por 100 €.</li>
<li>Entrega la cesta al gestor y recibe 1 participación del ETF (creación, valor 100 €).</li>
<li>Vende esa participación en la bolsa a 101,50 €.</li>
<li>Beneficio bruto: 1,50 € por participación.</li>
</ol>
<p><strong>Efecto en el mercado</strong> (aquí está la magia del arbitraje):</p>
<ul>
<li>Al <strong>comprar</strong> las acciones → sube el precio de la cesta → <strong>sube el NAV</strong>.</li>
<li>Al <strong>vender</strong> participaciones nuevas → aumenta la oferta del ETF → <strong>baja el precio de mercado</strong>.</li>
<li>Resultado: el NAV sube y el precio baja hasta encontrarse → <strong>la prima se cierra</strong>.</li>
</ul>
<p><strong>Caso B: el ETF cotiza con descuento (precio &lt; NAV).</strong></p>
<p><em>Situación</em>: NAV = 100 €, pero el ETF cotiza a 98,50 € (descuento del 1,5 %) porque hay pánico vendedor.</p>
<p>El AP hace el camino <strong>invertido</strong>:</p>
<ol>
<li>Compra participaciones del ETF en la bolsa por 98,50 €.</li>
<li>Entrega las participaciones al gestor y recibe la cesta de acciones (redención, valor 100 €).</li>
<li>Vende las acciones de la cesta en la bolsa por 100 €.</li>
<li>Beneficio bruto: 1,50 € por participación.</li>
</ol>
<p><strong>Efecto en el mercado</strong>:</p>
<ul>
<li>Al <strong>comprar</strong> participaciones → sube el precio del ETF.</li>
<li>Al <strong>vender</strong> las acciones de la cesta → baja el NAV.</li>
<li>Resultado: precio sube, NAV baja → <strong>el descuento se cierra</strong>.</li>
</ul>
<p><strong>El "anillo de tolerancia": por qué el spread no converge exactamente a cero.</strong></p>
<p>El proceso se repite <strong>hasta que ya no queda beneficio</strong>:</p>
<p><code>Beneficio del AP = |Spread| − Costes de transacción</code></p>
<ul>
<li>Mientras el spread sea <strong>mayor</strong> que los costes de operar (comisiones, bid-ask de las acciones, riesgo durante los segundos que dura la operación), el AP sigue arbitrando.</li>
<li>Cuando el spread cae <strong>por debajo</strong> de esos costes, ya no merece la pena → el arbitraje se detiene.</li>
</ul>
<p>Por eso el spread no converge exactamente a cero, sino que <strong>oscila en una banda estrecha</strong> alrededor de cero, cuyo ancho son los costes de transacción. Esa banda es la "estacionariedad" que comentamos: el spread no deriva, siempre vuelve a su media.</p>
<p><strong>Ejemplo numérico completo.</strong> Imagina un ETF que replica el IBEX 35, con costes de arbitraje totales de aproximadamente el 0,10 % (10 pb):</p>
<table border="1" cellpadding="6" cellspacing="0">
<thead><tr><th>Momento</th><th>Precio ETF</th><th>NAV</th><th>Spread</th><th>¿Qué pasa?</th></tr></thead>
<tbody>
<tr><td>09:00 (apertura, demanda fuerte)</td><td>101,00 €</td><td>100,00 €</td><td>+1,00 %</td><td>El AP compra la cesta (100 €), crea participaciones, vende a 101 € → beneficio 0,90 €</td></tr>
<tr><td>09:05</td><td>100,40 €</td><td>100,05 €</td><td>+0,35 %</td><td>Sigue habiendo beneficio → continúa el arbitraje</td></tr>
<tr><td>09:20</td><td>100,12 €</td><td>100,05 €</td><td>+0,07 %</td><td>Menor que el coste (0,10 %) → <strong>el arbitraje se detiene</strong></td></tr>
<tr><td>11:00 (venta en pánico)</td><td>99,00 €</td><td>100,00 €</td><td>−1,00 %</td><td>El AP compra ETF (99 €), redime por la cesta, vende acciones (100 €) → beneficio 0,90 €</td></tr>
<tr><td>11:20</td><td>99,95 €</td><td>100,00 €</td><td>−0,05 %</td><td>Dentro de la banda → se detiene</td></tr>
</tbody>
</table>
<p>El spread "respira" en torno a cero sin alejarse nunca más de lo que permiten los costes. Eso es un <strong>proceso estacionario con reversión a la media</strong>.</p>

<h4>6️⃣ ¿Cuándo falla el mecanismo? (spread no estacionario)</h4>
<p>El arbitraje solo funciona si el AP puede <strong>comprar o vender la cesta a un precio razonable</strong>. Falla cuando:</p>
<ol>
<li><strong>Mercados subyacentes cerrados</strong>: un ETF de acciones japonesas cotizando en Europa a las 15:00 (Tokio cerrado). El NAV es "del ayer"; nadie puede arbitrar → spreads amplios y persistentes hasta la reapertura de Asia.</li>
<li><strong>Activos ilíquidos en la cesta</strong>: ETFs de bonos high-yield, inmuebles o acciones microcap. Vender la cesta es lento y caro → el descuento puede persistir (ocurrió con varios ETFs en marzo de 2020).</li>
<li><strong>Límites al arbitraje</strong>: cuotas de creación agotadas, restricciones regulatorias o divisas con control de capitales (ETFs de India, China A-shares).</li>
<li><strong>ETFs apalancados/inversos</strong>: el NAV se calcula con derivados y swaps; el mecanismo existe, pero el valor no replica un índice comprable directamente.</li>
</ol>
<p>En esos casos el spread puede <strong>derivar</strong> (no estacionario) y el precio del ETF deja de reflejar fielmente el valor de sus activos.</p>

<h4>7️⃣ ¿Por qué importa para un inversor?</h4>
<table border="1" cellpadding="6" cellspacing="0">
<thead><tr><th>Situación</th><th>Implicación práctica</th></tr></thead>
<tbody>
<tr><td><strong>Spread pequeño y estacionario</strong></td><td>El ETF es <strong>barato y seguro de negociar</strong>: obtienes aproximadamente el valor real de los activos en cada transacción.</td></tr>
<tr><td><strong>Spread amplio</strong> (ETFs ilíquidos, mercados cerrados, crisis)</td><td>Puedes comprar con prima o vender con descuento, lo que supone una <strong>pérdida real</strong> aunque el activo subyacente no se mueva.</td></tr>
<tr><td><strong>Spread no estacionario o persistente</strong></td><td>Señal de <strong>alerta</strong>: posibles restricciones (carteras con activos que no cotizan, límites al arbitraje o ETFs apalancados/inversos con efecto <em>decay</em>).</td></tr>
</tbody>
</table>
<p><strong>Reglas operativas derivadas:</strong></p>
<ul>
<li>Opera ETFs con un spread habitual inferior a 10–20 pb.</li>
<li>Evita operar en la apertura (el spread está más ancho hasta que el arbitraje calibra los precios) ni cuando los mercados subyacentes están cerrados (ej.: ETF de Asia negociando en Europa).</li>
<li>Si haces backtesting, un spread estacionario justifica usar el NAV/índice como <em>proxy</em> del precio del ETF con poco error.</li>
</ul>

<h4>🔎 En una frase</h4>
<blockquote><p><strong>"El spread entre un ETF y su cesta es estacionario"</strong> = aunque el precio del ETF se desvíe puntualmente del valor de sus activos, esas desviaciones se corrigen automáticamente (gracias al arbitraje de creación/redención de los APs), por lo que la diferencia oscila de forma estable alrededor de cero —dentro de una banda delimitada por los costes de transacción— en lugar de crecer indefinidamente.</p></blockquote>